# Did LEVIATHAN filter the inversions?
So it doesn't seem like repeat-regions are too responsible for the false negatives (uncalled inversions). The next place to look is the `.candidates` files LEVIATHAN outputs. These files contain the list of all putatively called structural variants, which the program then filters based on some criteria to output the final called variant set. The question we're trying to address here is: **did LEVIATHAN identify the inversions initially and then filter them out?**

Using this python code, we parsed the leviathan candidates files to create a single file with all the candidate SVs across all samples and treatments.

```python
import glob

sizes = ["small", "medium", "large", "xl"]
depths = ["0.5", "2", "5", "10", "20"]
logdir = "logs/leviathan"

with open("simulated_data/called_sv/leviathan/all.candidates", "w") as candidates:
  for _size in sizes:
    for _depth in depths:
      _dir = f"simulated_data/called_sv/leviathan/{_size}/depth_{_depth}"
      _cand_files = [f"{_dir}/by_pop90/{logdir}/pop1.candidates"]
      for i in glob.glob(f"{_dir}/by_sample90/{logdir}/*candidates", recursive=True):
        _cand_files.append(i)
      for i in _cand_files:
        with open(i, "r") as _file:
          for line in _file:
            _samp = os.path.basename(i).replace(".candidates", "")
            candidates.write(f"{_samp}\t{_size}\t{_depth}\t" + line.lstrip())
```

Where the resulting file looks like:
```
sample_03	xl	20	2L	6083000	6083999	2L	6072000	6072999	7
sample_03	xl	20	3R	7155000	7155999	3R	7133000	7133999	8
sample_03	xl	20	3R	7155000	7155999	3R	7142000	7142999	8
sample_03	xl	20	3R	7155000	7155999	3R	7154000	7154999	34
```

As a sanity check, here are the top and bottom few lines of the resulting file

In [ ]:
%%bash
head simulated_data/called_sv/leviathan/all.candidates
tail simulated_data/called_sv/leviathan/all.candidates

pop1	small	0.5	2L	12481000	12481999	2L	12501000	12501999	4
pop1	small	0.5	2R	24097000	24097999	2R	24088000	24088999	5
pop1	small	0.5	2R	24097000	24097999	2R	24089000	24089999	5
pop1	small	0.5	3L	4491000	4491999	3L	4490000	4490999	10
pop1	small	0.5	3R	2832000	2832999	3R	2821000	2821999	4
pop1	small	0.5	3R	2832000	2832999	3R	2845000	2845999	6
pop1	small	0.5	3R	2832000	2832999	3R	2822000	2822999	6
pop1	small	0.5	3R	2832000	2832999	3R	2831000	2831999	12
pop1	small	0.5	3R	2832000	2832999	3R	2833000	2833999	11
pop1	small	0.5	3R	2832000	2832999	3R	2841000	2841999	5
sample_03	xl	20	2L	12351000	12351999	2L	12332000	12332999	8
sample_03	xl	20	2L	12351000	12351999	2L	12376000	12376999	7
sample_03	xl	20	2L	12351000	12351999	2L	12365000	12365999	9
sample_03	xl	20	2L	6083000	6083999	2L	6103000	6103999	7
sample_03	xl	20	2L	6083000	6083999	2L	6064000	6064999	9
sample_03	xl	20	2L	6083000	6083999	2L	6086000	6086999	13
sample_03	xl	20	2L	6083000	6083999	2L	6072000	6072999	7
sample_03	xl	20	3R	7155000	715

We'll continue this analysis in R